In [0]:
import json
import random
import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import current_timestamp

TARGET_RAW_TABLE = "payment_gateway_catalog.raw.raw_payment_payloads"
NUM_RECORDS = 100000

providers = ["razorpay", "dodo_payments", "slice"]

def generate_record():
    provider = random.choice(providers)
    event_id = f"evt_{provider}_{uuid.uuid4().hex[:10]}"
    timestamp = datetime.now(timezone.utc).isoformat()
    
    # Inject ~6% error cases
    is_error = random.random() < 0.06

    if provider == "razorpay":
        error_payload = {"code": "BANK_DOWNTIME", "description": "Issuer bank network unreachable."} if is_error else None
        return {
            "event_id": event_id,
            "provider": "razorpay",
            "timestamp": timestamp,
            "event_type": "payment.failed" if is_error else "payment.captured",
            "data": {
                "payment_id": f"pay_{uuid.uuid4().hex[:12]}",
                "order_id": f"order_{uuid.uuid4().hex[:10]}",
                "amount_subunits": random.randint(10000, 500000),
                "currency": "INR",
                "status": "failed" if is_error else "captured",
                "method": random.choice(["upi", "card", "netbanking"]),
                "vpa": "customer@okhdfcbank",
                "fee": 4998,
                "tax": 899,
                "error": error_payload
            }
        }
    
    elif provider == "dodo_payments":
        error_payload = {"code": "CARD_EXPIRED", "description": "The card provided has expired."} if is_error else None
        return {
            "event_id": event_id,
            "provider": "dodo_payments",
            "timestamp": timestamp,
            "event_type": "payment.failed" if is_error else "payment.succeeded",
            "data": {
                "payment_id": f"dodo_pay_{uuid.uuid4().hex[:8]}",
                "customer_id": f"cus_{uuid.uuid4().hex[:6]}",
                "amount_subunits": random.randint(500, 20000),
                "currency": "USD",
                "fx_rate_to_inr": round(random.uniform(83.0, 87.0), 2),
                "tax_jurisdiction": "US-CA",
                "vat_or_sales_tax_collected": 392,
                "status": "failed" if is_error else "succeeded",
                "error": error_payload
            }
        }
        
    else:  
        error_payload = {"code": "LIMIT_EXCEEDED", "description": "Customer requested transaction exceeding available pre-approved credit line limit."} if is_error else None
        return {
            "event_id": event_id,
            "provider": "slice",
            "timestamp": timestamp,
            "event_type": "transaction.failed" if is_error else "transaction.succeeded",
            "data": {
                "transaction_id": f"slc_tx_{uuid.uuid4().hex[:8]}",
                "account_id": f"slc_acc_{uuid.uuid4().hex[:5]}",
                "amount_subunits": random.randint(50000, 1000000),
                "currency": "INR",
                "tenure_months": random.choice([3, 6, 9, 12]),
                "merchant_category_code": "5411",
                "status": "failed" if is_error else "succeeded",
                "error": error_payload
            }
        }

records = [{"raw_payload": json.dumps(generate_record())} for _ in range(NUM_RECORDS)]

# Convert directly to DataFrame and write to Raw Managed Table
df = spark.createDataFrame(records).withColumn("_ingested_at", current_timestamp())
df.write.format("delta").mode("append").saveAsTable(TARGET_RAW_TABLE)

print(f"Directly ingested {NUM_RECORDS} raw JSON records into catalog table {TARGET_RAW_TABLE}")

In [0]:
df = spark.table(TARGET_RAW_TABLE)
df.display()
df.count()